##
**<font color="blueblueblueblue"><h3>👇  نصب کتابخانه‌ها</h3></font>**








In [ ]:

#@title #### **Install**
import base64
encoded_text = "Q3JlYXRlIGJ5IDogYWlnb2xkZW4="
decoded_text = base64.b64decode(encoded_text.encode()).decode()
print(decoded_text)
print('='*30)

!pip install -q -U google-genai yt-dlp pysrt pydub
!sudo apt-get update -qq
!sudo apt-get install -y -qq ffmpeg rubberband-cli nodejs
print('✅ همه چیز آماده‌ست (موتور Node.js نصب شد)!')

##
**<font color="blue"><h3>⚙️  تنظیمات عمومی</h3></font>**



In [ ]:

#@title #### **🔑 API KEY**
import os

GOOGLE_API_KEY = '' #@param {type:"string"}
#@markdown ---
#@markdown ####🌎 **زبان مقصد برای ترجمه**
Target_Language = 'fa' #@param ["fa", "en", "ar", "de", "fr", "es", "it", "tr", "ru", "zh-Hans", "ja", "ko", "hi", "pl", "pt-BR"]
#@markdown ---
#@markdown ####🗣️ **انتخاب گوینده**
Speaker_Voice = 'Charon' #@param ["Zephyr", "Puck", "Charon", "Kore", "Fenrir", "Leda", "Orus", "Aoede", "Callirrhoe", "Autonoe", "Enceladus", "Iapetus", "Umbriel", "Algieba", "Despina", "Erinome", "Algenib", "Rasalgethi", "Laomedeia", "Achernar", "Alnilam", "Schedar", "Gacrux", "Pulcherrima", "Achird", "Zubenelgenubi", "Vindemiatrix", "Sadachbia", "Sadaltager", "Sulafat"]
#@markdown ---
#@markdown ####✨ **لحن گویندگی**
Tone = 'عامیانه / گفتاری' #@param ["عامیانه / گفتاری", "رسمی / اخبار / کتابی", "آموزشی / آکادمیک", "پرانرژی / تبلیغاتی", "طبیعی / پیش‌فرض"]
#@markdown ---
Podcast_Mode = False #@param {type:"boolean"}
#@markdown ---
#@markdown ####🎚️ **صدای اصلی**
Keep_Original_Audio = True #@param {type:"boolean"}
Original_Audio_Volume = 0.1 #@param {type:"slider", min:0, max:1, step:0.05}

os.environ['GOOGLE_API_KEY'] = GOOGLE_API_KEY
print(f'✅ کلید تنظیم شد')
print(f'🌐 زبان مقصد: {Target_Language}')
print(f'🎙️ گوینده: {Speaker_Voice}')
print(f'🎭 لحن بیان: {Tone}')
print(f'📻 حالت پادکست: {"فعال" if Podcast_Mode else "غیرفعال"}')

In [ ]:

#@title ⚙️ تنظیمات پیشرفته زمان‌بندی و سینک
Allow_Timeline_Stretch = True #@param {type:"boolean"}
Max_Stretch_Percent = 8 #@param {type:"slider", min:0, max:25, step:1}
Balance_Speed_Across_Segments = True #@param {type:"boolean"}
Max_Speed_Factor = 1.5 #@param {type:"slider", min:1.0, max:2.0, step:0.1}

print("✅ تنظیمات زمان‌بندی و سینک ثبت شد:")
print(f" - کشش زمان‌بندی: {'فعال' if Allow_Timeline_Stretch else 'غیرفعال'} ({Max_Stretch_Percent}%)")
print(f" - تعادل سرعت: {'فعال' if Balance_Speed_Across_Segments else 'غیرفعال'} (حداکثر {Max_Speed_Factor}x)")

##
**<font color="redblueblack"><h3>  آپلود ویدیو یا صدا</h3></font>**



In [ ]:

#@title ####📂 **Select video**
import subprocess, os, glob, shutil
from google.colab import files

for f in glob.glob('input_video*') + glob.glob('yt_audio*') + ['audio.mp3', 'audio.srt', 'audio_translated.srt']:
    if os.path.exists(f):
        os.remove(f)
if os.path.exists('dubbing_project'):
    shutil.rmtree('dubbing_project')
if os.path.exists('audio_chunks'):
    shutil.rmtree('audio_chunks')

Upload_Method = 'یوتیوب' #@param ["یوتیوب", "فایل از دستگاه"]
YT_Link = '' #@param {type:"string"}

AUDIO_FILE = None
VIDEO_FILE = None
SRT_FILE = None
SUBTITLE_READY = False

# پارامترهای ضد بوت برای yt-dlp
YT_EXTRA_ARGS = ['--extractor-args', 'youtube:player_client=android,mweb,ios']

if Upload_Method == 'فایل از دستگاه':
    print('📁 فایل ویدیو یا صدا را آپلود کنید:')
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('❌ فایلی آپلود نشد!')
    fname = list(uploaded.keys())[0]
    ext = fname.rsplit('.', 1)[-1].lower()
    if ext in ['mp4', 'mkv', 'avi', 'mov', 'webm']:
        VIDEO_FILE = 'input_video.mp4'
        os.rename(fname, VIDEO_FILE)
        res = subprocess.run(['ffmpeg', '-y', '-i', VIDEO_FILE, '-vn', '-acodec', 'libmp3lame',
                              '-ab', '64k', '-ac', '1', 'audio.mp3'], capture_output=True, text=True)
        if res.returncode != 0:
            raise RuntimeError(f'❌ استخراج صدا از فایل آپلودی با خطا مواجه شد:\n{res.stderr}')
        AUDIO_FILE = 'audio.mp3'
        print('✅ ویدیو آپلود شد، صدا استخراج شد')
    else:
        mp3_name = 'audio.mp3'
        res = subprocess.run(['ffmpeg', '-y', '-i', fname, '-acodec', 'libmp3lame',
                              '-ab', '64k', '-ac', '1', mp3_name], capture_output=True, text=True)
        if res.returncode != 0:
            raise RuntimeError(f'❌ تبدیل فایل صوتی با خطا مواجه شد:\n{res.stderr}')
        AUDIO_FILE = mp3_name
        print('✅ فایل صوتی آماده شد')

else:
    if not YT_Link.strip():
        raise ValueError('❌ لینک یوتیوب وارد نشده است!')

    print('🔍 بررسی زیرنویس ویدیو...')
    check = subprocess.run(['yt-dlp', '--list-subs'] + YT_EXTRA_ARGS + [YT_Link], capture_output=True, text=True)
    if check.returncode != 0:
        # تلاش مجدد با کلاینت جایگزین
        check = subprocess.run(['yt-dlp', '--list-subs', '--extractor-args', 'youtube:player_client=tv,mweb', YT_Link], capture_output=True, text=True)
        if check.returncode != 0:
            raise RuntimeError(f'❌ دسترسی به لینک یوتیوب امکان‌پذیر نیست یا ویدیو دارای محدودیت است:\n{check.stderr}')
        else:
            YT_EXTRA_ARGS = ['--extractor-args', 'youtube:player_client=tv,mweb']

    has_sub = 'has no subtitles' not in check.stdout and 'Language' in check.stdout

    if has_sub:
        print('✅ زیرنویس یافت شد، در حال دانلود...')
        res_sub = subprocess.run(['yt-dlp', '--write-subs', '--write-auto-subs',
                                  '--sub-format', 'srt', '--convert-subs', 'srt',
                                  '--skip-download', '--output', 'yt_audio.%(ext)s'] + YT_EXTRA_ARGS + [YT_Link],
                                 capture_output=True, text=True)
        srt_files = [f for f in os.listdir('.') if f.startswith('yt_audio') and f.endswith('.srt')]
        if srt_files:
            os.rename(srt_files[0], 'audio.srt')
            SUBTITLE_READY = True
            SRT_FILE = 'audio.srt'
            print('✅ زیرنویس دانلود شد!')

    print('🎬 در حال دانلود ویدیو...')
    res_vid = subprocess.run(['yt-dlp', '-f', 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best',
                              '-o', 'input_video.%(ext)s'] + YT_EXTRA_ARGS + [YT_Link], capture_output=True, text=True)
    if res_vid.returncode != 0:
        raise RuntimeError(f'❌ دانلود ویدیو از یوتیوب ناموفق بود:\n{res_vid.stderr}')

    vfiles = glob.glob('input_video.*')
    if not vfiles:
        raise FileNotFoundError('❌ فایل ویدیوی دانلود شده یافت نشد!')

    if not vfiles[0].endswith('.mp4'):
        res_cv = subprocess.run(['ffmpeg', '-y', '-i', vfiles[0], '-c', 'copy', 'input_video.mp4'], capture_output=True, text=True)
        if res_cv.returncode != 0:
            raise RuntimeError(f'❌ تبدیل ویدیو به mp4 با خطا مواجه شد:\n{res_cv.stderr}')
        os.remove(vfiles[0])

    VIDEO_FILE = 'input_video.mp4'
    print('✅  با موفقیت پیدا شد.')

    if not SUBTITLE_READY:
        print('🎵 در حال استخراج صدا...')
        res_aud = subprocess.run(['yt-dlp', '--extract-audio', '--audio-format', 'mp3',
                                  '--audio-quality', '5', '--postprocessor-args', 'ExtractAudio:-ac 1',
                                  '-o', 'audio.%(ext)s'] + YT_EXTRA_ARGS + [YT_Link], capture_output=True, text=True)
        if res_aud.returncode != 0 or not os.path.exists('audio.mp3'):
            raise RuntimeError(f'❌ استخراج صدای یوتیوب با خطا مواجه شد:\n{res_aud.stderr}')

        AUDIO_FILE = 'audio.mp3'
        size_mb = os.path.getsize(AUDIO_FILE) / 1024 / 1024
        print(f'✅ صدا استخراج شد ({size_mb:.1f} MB)')

print('\n▶️ فرایند با موفقیت انجام شد. حالا سلول بعدی را اجرا کنید.')

####
**<font color="red"><h3> استخراج و ترجمه زیرنویس</h3></font>**



In [ ]:

#@title ##🎙️ **شروع استخراج**
import os, sys, re, time, math, subprocess
import pysrt
from concurrent.futures import ThreadPoolExecutor, as_completed
from google import genai
from google.genai import types
from pydub import AudioSegment

TRANS_INSTRUCTION = (
    "CRITICAL MANDATE FOR AUDIO TRANSCRIBING:\n"
    "1. LANGUAGE STRICTNESS: Transcribe EXACTLY in the ORIGINAL spoken audio language. STRICTLY DO NOT TRANSLATE into English or any other language!\n"
    "2. TIMING & SYNC: Ensure timestamps match natural speech boundaries exactly in standard SRT format. Start and end times must accurately reflect when each full sentence or phrase is spoken.\n"
    "3. HUMAN SPEECH ONLY: Transcribe ONLY actual spoken human words. COMPLETELY IGNORE and DO NOT include any non-speech sounds or descriptions such as [applause], [کف زدن], [music], [موسیقی], [wind], [صدای باد], [laughter], [خنده], background noise, sound effects, or any bracketed/parenthetical sound descriptions. If a segment contains no spoken words (only ambient sound, applause, music, or noise), SKIP it entirely — do not create a subtitle entry for it."
)

def get_audio_duration(file_path):
    audio = AudioSegment.from_file(file_path)
    return len(audio) / 1000.0

def split_audio_on_silence(file_path, chunk_target_sec=600):
    duration = get_audio_duration(file_path)
    if duration <= chunk_target_sec:
        return [(file_path, 0.0)]

    print(f"✂️ طول فایل {duration/60:.1f} دقیقه است. در حال برش هوشمند روی سکوت...")
    cmd = [
        'ffmpeg', '-i', file_path,
        '-af', 'silencedetect=noise=-30dB:d=0.5',
        '-f', 'null', '-'
    ]
    res = subprocess.run(cmd, capture_output=True, text=True)
    silences = re.findall(r'silence_start: ([\d\.]+)', res.stderr)
    silence_times = [float(s) for s in silences]

    os.makedirs('audio_chunks', exist_ok=True)
    chunks = []
    current_start = 0.0
    chunk_idx = 0

    while current_start < duration:
        target_time = current_start + chunk_target_sec
        if target_time >= duration:
            cut_time = duration
        else:
            candidates = [s for s in silence_times if current_start + 300 <= s <= current_start + chunk_target_sec + 60]
            cut_time = min(candidates, key=lambda x: abs(x - target_time)) if candidates else target_time

        chunk_path = f"audio_chunks/chunk_{chunk_idx:03d}.mp3"
        dur_segment = cut_time - current_start
        subprocess.run([
            'ffmpeg', '-y', '-ss', str(current_start), '-t', str(dur_segment),
            '-i', file_path, '-acodec', 'copy', chunk_path
        ], capture_output=True)

        chunks.append((chunk_path, current_start))
        print(f"  📌 تکه {chunk_idx+1}: از {current_start:.1f}s تا {cut_time:.1f}s")
        current_start = cut_time
        chunk_idx += 1

    return chunks

def transcribe_chunk_with_fallback(chunk_info):
    chunk_path, offset_sec = chunk_info
    client = genai.Client(api_key=os.environ['GOOGLE_API_KEY'])
    models = ['gemini-3.7-flash', 'gemini-3.6-flash', 'gemini-3.5-flash-lite']

    with open(chunk_path, 'rb') as f:
        audio_bytes = f.read()

    for model_name in models:
        for attempt in range(1, 4):
            try:
                print(f"🚀 در حال پردازش {os.path.basename(chunk_path)} با {model_name} (تلاش {attempt})...")
                response = client.models.generate_content(
                    model=model_name,
                    contents=[
                        types.Part.from_bytes(data=audio_bytes, mime_type='audio/mp3'),
                        TRANS_INSTRUCTION
                    ]
                )
                srt_text = response.text.strip()
                if "1\n" in srt_text or "00:" in srt_text:
                    return srt_text, offset_sec
                else:
                    print(f"   ⚠️ فرمت نامعتبر از {model_name}: {srt_text[:100]}")
            except Exception as e:
                print(f"   ⚠️ خطا با {model_name}: {e}")
                time.sleep(2 * attempt)
    return "", offset_sec

def shift_srt_timestamps(srt_text, offset_sec):
    lines = srt_text.strip().split('\n')
    new_lines = []
    pattern = re.compile(r'(\d{2}):(\d{2}):(\d{2})[,.](\d{3})\s*-->\s*(\d{2}):(\d{2}):(\d{2})[,.](\d{3})')

    def add_offset(h, m, s, ms):
        tot = (int(h)*3600 + int(m)*60 + int(s)) + int(ms)/1000.0 + offset_sec
        nh, rem = divmod(tot, 3600)
        nm, ns = divmod(rem, 60)
        nms = int((ns - int(ns)) * 1000)
        return f"{int(nh):02d}:{int(nm):02d}:{int(ns):02d},{nms:03d}"

    for line in lines:
        match = pattern.search(line)
        if match:
            start_str = add_offset(*match.groups()[:4])
            end_str = add_offset(*match.groups()[4:])
            new_lines.append(f"{start_str} --> {end_str}")
        else:
            new_lines.append(line)
    return "\n".join(new_lines)

def analyze_wpm(srt_file):
    if not os.path.exists(srt_file): return
    subs = pysrt.open(srt_file, encoding='utf-8')
    if not subs: return

    total_words = 0
    total_seconds = 0
    wpm_list = []
    high_speed_count = 0

    for sub in subs:
        words = len(sub.text.split())
        duration = (sub.end.ordinal - sub.start.ordinal) / 1000.0
        if duration <= 0: continue
        wpm = (words / duration) * 60
        wpm_list.append(wpm)
        total_words += words
        total_seconds += duration
        if wpm > 150:
            high_speed_count += 1

    avg_wpm = (total_words / total_seconds) * 60 if total_seconds > 0 else 0
    print("\n" + "="*50)
    print("📊 **گزارش آنالیز سرعت گفتار (WPM Log)**")
    print(f"🔹 میانگین کل سرعت گفتار: {avg_wpm:.1f} کلمه در دقیقه")

    if avg_wpm <= 110:
        print("🟢 وضعیت: آهسته و شمرده (سبز)")
    elif avg_wpm <= 150:
        print("🔵 وضعیت: طبیعی و روان (آبی)")
    elif avg_wpm <= 180:
        print("🟡 وضعیت: سریع و متراکم (زرد)")
    else:
        print("🔴 وضعیت: بسیار سریع و فشرده (قرمز)")

    if high_speed_count > 0:
        percent = (high_speed_count / len(subs)) * 100
        print(f"\n⚠️ **هشدار:** {percent:.1f}% دیالوگ‌ها سرعت زرد یا قرمز دارند.")
        print("👉 **توصیه:** قبل از ترجمه، تیک 'خلاصه‌سازی ترجمه' را در سلول ۲ بزنید.")
    print("="*50 + "\n")

if SUBTITLE_READY and SRT_FILE:
    print('✅ زیرنویس از قبل آماده‌ست.')
    analyze_wpm(SRT_FILE)
elif AUDIO_FILE is None:
    print('❌ ابتدا سلول ورودی را اجرا کنید!')
else:
    SRT_FILE = 'audio.srt'
    chunks = split_audio_on_silence(AUDIO_FILE, chunk_target_sec=600)

    results = []
    with ThreadPoolExecutor(max_workers=len(chunks)) as executor:
        futures = [executor.submit(transcribe_chunk_with_fallback, chunk) for chunk in chunks]
        for future in as_completed(futures):
            results.append(future.result())

    results.sort(key=lambda x: x[1])

    final_srt_blocks = []
    global_index = 1
    for raw_srt, offset in results:
        shifted = shift_srt_timestamps(raw_srt, offset)
        subs = pysrt.from_string(shifted)
        for sub in subs:
            sub.index = global_index
            global_index += 1
            final_srt_blocks.append(str(sub))

    with open(SRT_FILE, 'w', encoding='utf-8') as f:
        f.write("\n\n".join(final_srt_blocks))

    print(f'✅ زیرنویس کامل ذخیره شد: {SRT_FILE}')
    analyze_wpm(SRT_FILE)

In [ ]:

#@title ##🌐 **ترجمه زیرنویس**
import pysrt, os, time, re
from concurrent.futures import ThreadPoolExecutor, as_completed
from google import genai

#@markdown ####  **خلاصه‌سازی ترجمه**
Enable_Summarization = False #@param {type:"boolean"}
print(f'📝 خلاصه‌سازی ترجمه: {"فعال" if Enable_Summarization else "غیرفعال"}')

def translate_batch(batch_subs, target_lang, enable_summary, tone_option):
    client = genai.Client(api_key=os.environ['GOOGLE_API_KEY'])
    models = ['gemini-3.5-flash-lite', 'gemini-3.1-flash-lite']

    texts = [f"{i+1}. {sub.text}" for i, sub in enumerate(batch_subs)]
    combined_text = "\n".join(texts)

    tone_prompts = {
        "عامیانه / گفتاری": "Use casual, everyday spoken phrasing with natural contractions, like talking to a friend.",
        "رسمی / اخبار / کتابی": "Use formal, precise written language suitable for news or literary narration.",
        "آموزشی / آکادمیک": "Use clear, structured, instructional phrasing suitable for an educational lecture.",
        "پرانرژی / تبلیغاتی": "Use energetic, punchy, exclamatory phrasing suitable for an advertisement.",
        "طبیعی / پیش‌فرض": "Use natural, balanced everyday phrasing."
    }
    tone_prompt = tone_prompts.get(tone_option, tone_prompts["طبیعی / پیش‌فرض"])

    summary_prompt = "Keep translations concise and shorter while preserving the full meaning." if enable_summary else ""
    prompt = f"Translate the following subtitles into target language '{target_lang}'. {tone_prompt} {summary_prompt} Maintain line numbering strictly.\n\n{combined_text}"

    for model_name in models:
        try:
            res = client.models.generate_content(model=model_name, contents=prompt)
            lines = res.text.strip().split('\n')
            translated = [re.sub(r'^\d+\.\s*', '', line) for line in lines if line.strip()]
            if len(translated) == len(batch_subs):
                return translated
        except Exception:
            time.sleep(1)

    return [sub.text for sub in batch_subs]

# مسیر فایل اصلی زیرنویس ثابت و مستقل از SRT_FILE، تا اجراهای مکرر همیشه از روی زیرنویس اصلی ترجمه کنن
ORIGINAL_SRT_FILE = 'audio.srt'

if os.path.exists(ORIGINAL_SRT_FILE):
    print("🌐 در حال ترجمه موازی زیرنویس...")
    subs = pysrt.open(ORIGINAL_SRT_FILE, encoding='utf-8')
    batch_size = 20
    batches = [subs[i:i + batch_size] for i in range(0, len(subs), batch_size)]

    translated_subs = []
    with ThreadPoolExecutor(max_workers=5) as executor:
        futures = [
            executor.submit(translate_batch, b, Target_Language, Enable_Summarization, Tone)
            for b in batches
        ]
        for future in futures:
            translated_subs.extend(future.result())

    for i, t_text in enumerate(translated_subs):
        if i < len(subs):
            subs[i].text = t_text

    TRANSLATED_SRT = 'audio_translated.srt'
    subs.save(TRANSLATED_SRT, encoding='utf-8')
    SRT_FILE = TRANSLATED_SRT
    print(f"✅ ترجمه کامل شد و در {SRT_FILE} ذخیره شد.")
else:
    print("⚠️ فایل اصلی زیرنویس (audio.srt) پیدا نشد.")

##
**<font color="yeblue"><h3>   شروع دوبله   </h3></font>**



In [ ]:

#@title ## 🎬 **دوبله هوشمند**
import asyncio, os, subprocess, struct, shutil
import pysrt
from google import genai
from google.genai import types
from pydub import AudioSegment

os.makedirs('dubbing_project/dubbed_segments', exist_ok=True)

def pcm_to_wav(pcm_data, sample_rate=24000):
    num_channels = 1
    bits_per_sample = 16
    byte_rate = sample_rate * num_channels * bits_per_sample // 8
    block_align = num_channels * bits_per_sample // 8
    data_size = len(pcm_data)
    header = struct.pack('<4sI4s4sIHHIIHH4sI',
        b'RIFF', 36 + data_size, b'WAVE', b'fmt ', 16, 1,
        num_channels, sample_rate, byte_rate, block_align,
        bits_per_sample, b'data', data_size)
    return header + pcm_data

async def send_text_turn(session, text_to_speak, timeout=20):
    # 🛠️ روش چندگانه برای سازگاری کامل با تمامی نسخه‌های SDK گوگل
    try:
        # روش ۱: ارسال مستقیم متن (استاندارد اصلی SDK)
        await session.send(input=text_to_speak, end_of_turn=True)
    except Exception:
        try:
            # روش ۲: ارسال با شیء LiveClientContent
            await session.send(
                types.LiveClientContent(
                    turns=[types.Content(role="user", parts=[types.Part.from_text(text=text_to_speak)])],
                    end_of_turn=True
                )
            )
        except Exception:
            # روش ۳: ارسال با ساختار متنی مستقیم
            await session.send(
                types.Content(
                    role="user",
                    parts=[types.Part.from_text(text=text_to_speak)]
                ),
                end_of_turn=True
            )

    received_audio = bytearray()
    try:
        async with asyncio.timeout(timeout):
            async for response in session.receive():
                if response.server_content:
                    if response.server_content.model_turn:
                        for part in response.server_content.model_turn.parts:
                            if part.inline_data and part.inline_data.data:
                                received_audio.extend(part.inline_data.data)
                if response.server_content and response.server_content.turn_complete:
                    break
    except asyncio.TimeoutError:
        pass

    return bytes(received_audio)

async def process_batch(batch_items, client, model, config, raw_paths, natural_ms):
    async with client.aio.live.connect(model=model, config=config) as session:
        for i, text_to_speak, subtitle_duration in batch_items:
            raw_path = raw_paths[i]
            if not text_to_speak or not text_to_speak.strip():
                AudioSegment.silent(duration=int(subtitle_duration*1000)).export(raw_path, format='wav')
                natural_ms[i] = int(subtitle_duration*1000)
                continue

            for attempt in range(1, 4):
                try:
                    pcm_audio = await send_text_turn(session, text_to_speak)
                    if not pcm_audio:
                        raise RuntimeError('پاسخ صوتی دریافت نشد (پاسخ خالی).')

                    wav_data = pcm_to_wav(pcm_audio, sample_rate=24000)
                    with open(raw_path, 'wb') as f:
                        f.write(wav_data)

                    natural_ms[i] = len(AudioSegment.from_file(raw_path))
                    print(f'  ✅ [{i+1}] خوانده شد: "{text_to_speak[:30]}..."')
                    break
                except Exception as e:
                    if attempt == 3:
                        AudioSegment.silent(duration=int(subtitle_duration*1000)).export(raw_path, format='wav')
                        natural_ms[i] = int(subtitle_duration*1000)
                        print(f'  ⚠️ [{i+1}] خطا: {e}')
                    else:
                        await asyncio.sleep(1.5)

def build_system_instruction(target_lang, tone_option):
    base_instruction = (
        "You are a professional text-to-speech engine. Your ONLY job is to speak the user's input text "
        "word for word in their language, with accurate pronunciation. "
        "You MUST NOT add any introduction, explanations, prefaces, filler, or ending remarks. "
        "Speak only the exact text given."
    )

    tone_instructions = {
        "عامیانه / گفتاری": " Use a relaxed, casual, conversational tone, like speaking naturally to a friend.",
        "رسمی / اخبار / کتابی": " Use a highly formal, serious, and professional tone suitable for news broadcasting or official narration.",
        "آموزشی / آکادمیک": " Use a clear, well-paced, instructional, and articulate tone suitable for online courses or educational lectures.",
        "پرانرژی / تبلیغاتی": " Use a high-energy, enthusiastic, engaging, and dynamic delivery suitable for commercials.",
        "طبیعی / پیش‌فرض": " Use a natural, balanced native speaking tone."
    }

    selected_tone_prompt = tone_instructions.get(tone_option, tone_instructions["طبیعی / پیش‌فرض"])
    full_instruction = base_instruction + selected_tone_prompt

    if str(target_lang).lower() in ['fa', 'farsi', 'persian']:
        full_instruction += (
            " Since the target language is Persian (Farsi), you MUST speak in a standard Tehrani accent "
            "(لهجه معیار تهرانی) as used in Tehran, Iran. Avoid any Afghan (Dari) or Tajik pronunciations, "
            "vocabulary, or regional accents completely."
        )

    return full_instruction

async def run_dubbing():
    global SRT_FILE, Target_Language, Speaker_Voice, Tone, Podcast_Mode
    global STRETCH_FACTOR
    STRETCH_FACTOR = 1.0

    user_tone = globals().get('Tone', 'طبیعی / پیش‌فرض')
    user_voice = globals().get('Speaker_Voice', 'Charon')
    user_lang = globals().get('Target_Language', 'fa')
    Podcast_Mode_ = globals().get('Podcast_Mode', False)
    Allow_Timeline_Stretch_ = globals().get('Allow_Timeline_Stretch', False)
    Max_Stretch_Percent_ = globals().get('Max_Stretch_Percent', 10)
    Balance_Speed_Across_Segments_ = globals().get('Balance_Speed_Across_Segments', True)
    Max_Speed_Factor_ = globals().get('Max_Speed_Factor', 1.4)

    srt_path = SRT_FILE if 'SRT_FILE' in globals() and SRT_FILE and os.path.exists(SRT_FILE) else 'audio_translated.srt'
    if not os.path.exists(srt_path):
        srt_path = 'audio.srt'

    if not os.path.exists(srt_path):
        print('❌ فایل زیرنویس یافت نشد!')
        return

    subs = pysrt.open(srt_path, encoding='utf-8')
    n = len(subs)

    windows_ms = [None] * n
    natural_ms = [None] * n
    raw_paths = [None] * n
    pending = []

    for i, sub in enumerate(subs):
        start_ms = (sub.start.hours*3600 + sub.start.minutes*60 + sub.start.seconds)*1000 + sub.start.milliseconds
        end_ms = (sub.end.hours*3600 + sub.end.minutes*60 + sub.end.seconds)*1000 + sub.end.milliseconds
        subtitle_duration = (end_ms - start_ms) / 1000.0

        if i < n - 1:
            next_sub = subs[i+1]
            next_start_ms = (next_sub.start.hours*3600 + next_sub.start.minutes*60 + next_sub.start.seconds)*1000 + next_sub.start.milliseconds
            window_ms = next_start_ms - start_ms
        else:
            window_ms = end_ms - start_ms

        raw_path = f'dubbing_project/dubbed_segments/raw_{i+1}.wav'
        windows_ms[i] = window_ms
        raw_paths[i] = raw_path

        if os.path.exists(raw_path) and os.path.getsize(raw_path) > 100:
            try:
                natural_ms[i] = len(AudioSegment.from_file(raw_path))
                print(f'  ♻️ [{i+1}/{n}] قبلاً تولید شده بود، رد شد.')
            except Exception:
                pending.append((i, sub.text, subtitle_duration))
        else:
            pending.append((i, sub.text, subtitle_duration))

    if pending:
        sys_instruction = build_system_instruction(user_lang, user_tone)
        api_key = os.environ.get('GOOGLE_API_KEY')
        if not api_key:
            print("❌ کلید GOOGLE_API_KEY یافت نشد. لطفاً سلول ۲ را مجدداً اجرا کنید.")
            return

        client = genai.Client(api_key=api_key)
        model = 'gemini-3.1-flash-live-preview'

        config = types.LiveConnectConfig(
            response_modalities=['AUDIO'],
            system_instruction=types.Content(parts=[types.Part.from_text(text=sys_instruction)]),
            speech_config=types.SpeechConfig(
                voice_config=types.VoiceConfig(
                    prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name=user_voice)
                )
            )
        )

        NUM_SESSIONS = 3
        chunk_size = -(-len(pending) // NUM_SESSIONS)
        batches = [pending[i:i + chunk_size] for i in range(0, len(pending), chunk_size)]

        print(f'🚀 مرحله ۱ از ۲: گویندگی {len(pending)} دیالوگ با {len(batches)} سشن موازی، گوینده [{user_voice}]...')
        tasks = [process_batch(b, client, model, config, raw_paths, natural_ms) for b in batches]
        await asyncio.gather(*tasks)

    if any(v is None for v in natural_ms):
        print("❌ برخی از دیالوگ‌ها با خطا مواجه شدند. متن خطای بالا را بررسی کنید.")
        return

    idx_borrowable = list(range(n - 1)) if n > 1 else []
    total_natural = sum(natural_ms[i] for i in idx_borrowable)
    total_window  = sum(windows_ms[i] for i in idx_borrowable)
    G = (total_natural / total_window) if total_window > 0 else 1.0

    if Allow_Timeline_Stretch_ and G > 1.0 and not Podcast_Mode_:
        max_stretch = 1 + (Max_Stretch_Percent_ / 100.0)
        STRETCH_FACTOR = min(G, max_stretch)
    if STRETCH_FACTOR > 1.0:
        windows_ms = [w * STRETCH_FACTOR if i in idx_borrowable else w for i, w in enumerate(windows_ms)]

    print(f"📐 نسبت طول دوبله به زمان اصلی (G): {G:.2f}  →  یعنی صدا حدود {(G-1)*100:.0f}٪ بیشتر از زمان اصلی فضا لازم داره")

    R = (total_natural / total_window) if total_window > 0 else 1.0
    base_factor = min(R, 1.3) if (Balance_Speed_Across_Segments_ and R > 1.0) else 1.0
    hard_cap = Max_Speed_Factor_ if Balance_Speed_Across_Segments_ else 1.4

    print("⚡ مرحله ۲ از ۲: تنظیم زمان‌بندی و سینک صدا...")
    for i in range(n):
        raw_path = raw_paths[i]
        final_path = f'dubbing_project/dubbed_segments/dub_{i+1}.wav'

        if Podcast_Mode_:
            shutil.copy(raw_path, final_path)
            continue

        adjusted_ms = natural_ms[i] / base_factor
        local_factor = adjusted_ms / windows_ms[i] if windows_ms[i] > 0 else 1.0
        final_factor = max(1.0, min(base_factor * max(1.0, local_factor), hard_cap))

        if final_factor <= 1.001:
            shutil.copy(raw_path, final_path)
        else:
            try:
                if not os.path.exists(raw_path) or os.path.getsize(raw_path) < 100:
                    shutil.copy(raw_path, final_path)
                    continue

                res = subprocess.run([
                    'ffmpeg', '-y', '-i', raw_path,
                    '-filter:a', f'rubberband=tempo={final_factor:.3f}',
                    final_path
                ], capture_output=True, text=True)

                if res.returncode != 0:
                    res_alt = subprocess.run([
                        'ffmpeg', '-y', '-i', raw_path,
                        '-filter:a', f'atempo={final_factor:.3f}',
                        final_path
                    ], capture_output=True, text=True)
                    if res_alt.returncode != 0:
                        shutil.copy(raw_path, final_path)
            except Exception:
                shutil.copy(raw_path, final_path)

await run_dubbing()

In [ ]:

#@title ##🎞️ **ترکیب نهایی و دانلود**
import subprocess, os, glob, shutil
import pysrt
from pydub import AudioSegment
from google.colab import files
from IPython.display import display, HTML

try:
    STRETCH_FACTOR
except NameError:
    STRETCH_FACTOR = 1.0

#@markdown ### **تأخیر جبرانی شروع دوبله**
Enable_Start_Delay = True #@param {type:"boolean"}
START_DELAY_MS = 150 if Enable_Start_Delay else 0

if Podcast_Mode:
    seg_files = sorted(
        glob.glob('dubbing_project/dubbed_segments/dub_*.wav'),
        key=lambda f: int(f.split('dub_')[1].split('.')[0])
    )
    if seg_files:
        combined = AudioSegment.empty()
        for sf in seg_files:
            combined += AudioSegment.from_file(sf)
        output_file = 'podcast_dubbed.mp3'
        combined.export(output_file, format='mp3', bitrate='192k')
        files.download(output_file)

else:
    if not VIDEO_FILE or not os.path.exists(VIDEO_FILE):
        subs = pysrt.open(SRT_FILE, encoding='utf-8')
        total_ms = (subs[-1].end.hours*3600 + subs[-1].end.minutes*60 +
                    subs[-1].end.seconds)*1000 + subs[-1].end.milliseconds
        total_ms = int(total_ms * STRETCH_FACTOR)
        final_audio = AudioSegment.silent(duration=total_ms)
        for i, sub in enumerate(subs):
            seg_path = f'dubbing_project/dubbed_segments/dub_{i+1}.wav'
            if os.path.exists(seg_path):
                start_ms = int(((sub.start.hours*3600 + sub.start.minutes*60 + sub.start.seconds)*1000 + sub.start.milliseconds) * STRETCH_FACTOR) + START_DELAY_MS
                seg_audio = AudioSegment.from_file(seg_path)
                final_audio = final_audio.overlay(seg_audio, position=start_ms)
        output_file = 'dubbed_audio.mp3'
        final_audio.export(output_file, format='mp3', bitrate='192k')
        files.download(output_file)
    else:
        subs = pysrt.open(SRT_FILE, encoding='utf-8')
        orig_tempo = 1/STRETCH_FACTOR if STRETCH_FACTOR > 1 else 1.0

        if Keep_Original_Audio:
            filter_parts = [f'[0:a]volume={Original_Audio_Volume},atempo={orig_tempo}[orig]']
        else:
            filter_parts = ['[0:a]volume=0[orig]']

        input_args = ['-i', VIDEO_FILE]
        valid_segs = []

        for i, sub in enumerate(subs):
            seg_path = f'dubbing_project/dubbed_segments/dub_{i+1}.wav'
            if os.path.exists(seg_path):
                start_ms = int(((sub.start.hours*3600 + sub.start.minutes*60 +
                            sub.start.seconds)*1000 + sub.start.milliseconds) * STRETCH_FACTOR) + START_DELAY_MS
                filter_parts.append(f'[{len(valid_segs)+1}:a]adelay={start_ms}|{start_ms}[a{i+1}]')
                input_args += ['-i', seg_path]
                valid_segs.append(i+1)

        if valid_segs:
            mix_inputs = '[orig]' + ''.join(f'[a{i}]' for i in valid_segs)
            filter_parts.append(f'{mix_inputs}amix=inputs={len(valid_segs)+1}:normalize=0[aout]')

            if STRETCH_FACTOR > 1.0:
                filter_parts.append(f'[0:v]setpts={STRETCH_FACTOR}*PTS[vout]')
                video_map = ['-map', '[vout]']
                vcodec = 'libx264'
            else:
                video_map = ['-map', '0:v']
                vcodec = 'copy'

            filter_complex = ';'.join(filter_parts)
            output_file = 'final_dubbed_video.mp4'
            cmd = ['ffmpeg', '-y'] + input_args + ['-filter_complex', filter_complex] + video_map + [
                '-map', '[aout]', '-c:v', vcodec, '-c:a', 'aac', output_file
            ]
            result = subprocess.run(cmd, capture_output=True, text=True)
            if result.returncode == 0:
                files.download(output_file)

image_url = 'https://huggingface.co/Toolsai/dubtest/resolve/main/newgolden.png'
yt_url = 'https://youtube.com/@aigolden'
display(HTML(f'''
<div style="text-align:center;border:2px solid #e0e0e0;padding:15px;border-radius:12px;background:#f9f9f9;max-width:350px;margin:auto">
  <a href="{yt_url}" target="_blank">
    <img src="{image_url}" style="max-width:100%;border-radius:8px">
  </a>
  <p style="font-family:Vazir,sans-serif;margin-top:10px">برای آموزش‌های بیشتر ما را در یوتیوب دنبال کنید</p>
  <a href="{yt_url}" target="_blank" style="background:#FF0000;color:white;padding:10px 20px;border-radius:8px;text-decoration:none;font-weight:bold">🚀 دنبال کردن</a>
</div>
'''))

------

In [ ]:

#@title 🧹 پاکسازی فایل‌های جلسه
!rm -rf /content/*
print('✅ پاکسازی انجام شد')